# EchoJEPA video embedding PCA visualization

This notebook loads a trained EchoJEPA checkpoint, extracts patch tokens from a video clip, and visualizes them with a DINO-style PCA color mapping.

## Temporal plan

We do **not** collapse the video into a single embedding immediately. Instead, we use the fact that the backbone is spatiotemporal in three complementary ways:

1. **Per-tubelet PCA color maps** — map the final patch-token cloud to RGB using PCA, then overlay those colors back onto each video segment.
2. **Temporal change maps** — compare embeddings from adjacent tubelets so motion and changing anatomy become visible, not just static appearance.
3. **Trajectory plots** — pool tokens per tubelet and project them to 2D to see how the clip moves through representation space over time.

For stable colors across multiple clips, fit PCA on a **bank of tokens from several videos** from the same view/domain, then reuse that PCA basis for every clip. If you only fit on one clip, the colors are still useful, but they become clip-relative rather than globally comparable.

In [ ]:
from pathlib import Path
import os
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from decord import VideoReader, cpu
from sklearn.decomposition import PCA

try:
    from src.datasets.video_dataset import VideoDataset
except Exception as exc:
    VideoDataset = None
    print(f'VideoDataset import skipped: {exc}')

from src.datasets.utils.video.transforms import CenterCrop, Compose, Normalize, Resize
from src.datasets.utils.video.volume_transforms import ClipToTensor
from src.models import vision_transformer as vit
from src.utils.checkpoint_loader import robust_checkpoint_loader

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(0)
torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

In [ ]:
IMAGENET_DEFAULT_MEAN = (0.485, 0.456, 0.406)
IMAGENET_DEFAULT_STD = (0.229, 0.224, 0.225)

# --- user-configurable inputs ---
CHECKPOINT_PATH = '/path/to/echojepa_checkpoint.pt'
CHECKPOINT_KEY = 'target_encoder'   # often 'target_encoder' for EchoJEPA inference checkpoints
MODEL_NAME = 'vit_giant_xformers_rope'
IMG_SIZE = 384
NUM_FRAMES = 16
TUBELET_SIZE = 2
PATCH_SIZE = 16
VIDEO_SOURCE = '/path/to/video_or_annotations.csv'
SAMPLE_INDEX = 0
FRAME_STEP = 2
NUM_CLIPS = 1
RANDOM_CLIP_SAMPLING = False
ALLOW_CLIP_OVERLAP = False

print('device:', device)
print('checkpoint:', CHECKPOINT_PATH)
print('video source:', VIDEO_SOURCE)

In [ ]:
def build_pt_video_transform(img_size: int):
    short_side_size = int(256.0 / 224 * img_size)
    return Compose([
        Resize(short_side_size, interpolation='bilinear'),
        CenterCrop(size=(img_size, img_size)),
        ClipToTensor(),
        Normalize(mean=IMAGENET_DEFAULT_MEAN, std=IMAGENET_DEFAULT_STD),
    ])

def strip_prefixes(state_dict):
    for prefix in ('module.', 'backbone.'):  # common training / DDP prefixes
        state_dict = {k.replace(prefix, ''): v for k, v in state_dict.items()}
    return state_dict

def load_encoder_from_checkpoint(model, checkpoint_path: str, checkpoint_key: str):
    ckpt = robust_checkpoint_loader(checkpoint_path, map_location='cpu')
    print('checkpoint keys:', list(ckpt.keys()))
    if checkpoint_key not in ckpt:
        fallback_keys = ['target_encoder', 'encoder']
        for key in fallback_keys:
            if key in ckpt:
                checkpoint_key = key
                break
    if checkpoint_key not in ckpt:
        raise KeyError(f'Could not find encoder weights in checkpoint. Tried {checkpoint_key!r}. Available keys: {list(ckpt.keys())}')

    state_dict = strip_prefixes(ckpt[checkpoint_key])
    msg = model.load_state_dict(state_dict, strict=False)
    print(f'Loaded {checkpoint_key!r} with msg: {msg}')
    return ckpt

def make_dataset(video_source: str, frames_per_clip: int, frame_step: int, num_clips: int = 1,
                 random_clip_sampling: bool = False, allow_clip_overlap: bool = False):
    if VideoDataset is None:
        return None

    if video_source.endswith('.csv') or video_source.endswith('.npy'):
        data_paths = video_source
        temp_csv = None
    else:
        tmp_dir = Path(tempfile.gettempdir())
        temp_csv = tmp_dir / 'echojepa_single_video.csv'
        temp_csv.write_text(f'{video_source} 0\n')
        data_paths = str(temp_csv)

    ds = VideoDataset(
        data_paths=data_paths,
        frames_per_clip=frames_per_clip,
        frame_step=frame_step,
        num_clips=num_clips,
        transform=None,
        shared_transform=None,
        random_clip_sampling=random_clip_sampling,
        allow_clip_overlap=allow_clip_overlap,
        filter_short_videos=False,
        filter_long_videos=int(10**9),
        duration=None,
        fps=None,
    )
    return ds

def load_raw_clip_from_source(video_source: str, sample_index: int = 0):
    ds = make_dataset(
        video_source=video_source,
        frames_per_clip=NUM_FRAMES,
        frame_step=FRAME_STEP,
        num_clips=NUM_CLIPS,
        random_clip_sampling=RANDOM_CLIP_SAMPLING,
        allow_clip_overlap=ALLOW_CLIP_OVERLAP,
    )
    if ds is not None:
        buffer, label, clip_indices = ds[sample_index]
        raw_clip = buffer[0] if isinstance(buffer, list) else buffer
        clip_indices = clip_indices[0] if isinstance(clip_indices, list) else clip_indices
        return raw_clip, clip_indices, label

    vr = VideoReader(video_source, num_threads=-1, ctx=cpu(0))
    frame_idx = np.arange(0, min(len(vr), NUM_FRAMES * FRAME_STEP), FRAME_STEP)[:NUM_FRAMES]
    raw_clip = vr.get_batch(frame_idx).asnumpy()
    return raw_clip, frame_idx, 0

def prepare_clip_for_model(raw_clip: np.ndarray, img_size: int):
    video = torch.from_numpy(raw_clip).permute(0, 3, 1, 2)  # T, C, H, W
    pt_transform = build_pt_video_transform(img_size)
    clip = pt_transform(video).unsqueeze(0)  # B, C, T, H, W
    return clip

def reshape_tokens_to_grid(tokens: torch.Tensor, clip: torch.Tensor, patch_size: int, tubelet_size: int):
    # tokens: [B, N, D]
    _, _, T, H, W = clip.shape
    t = T // tubelet_size
    h = H // patch_size
    w = W // patch_size
    assert tokens.shape[1] == t * h * w, (tokens.shape, t, h, w)
    return tokens.reshape(tokens.shape[0], t, h, w, tokens.shape[-1])

def fit_rgb_pca(token_bank: np.ndarray):
    pca = PCA(n_components=3, random_state=0)
    pca.fit(token_bank)
    return pca

def pca_to_rgb(token_grid: np.ndarray, pca: PCA):
    flat = token_grid.reshape(-1, token_grid.shape[-1])
    rgb = pca.transform(flat).reshape(*token_grid.shape[:-1], 3)
    rgb = rgb - rgb.min(axis=(0, 1, 2), keepdims=True)
    rgb = rgb / (rgb.max(axis=(0, 1, 2), keepdims=True) + 1e-6)
    return rgb

def upsample_grid(grid: np.ndarray, out_size: int):
    x = torch.from_numpy(grid).permute(0, 3, 1, 2).float()
    x = F.interpolate(x, size=(out_size, out_size), mode='bilinear', align_corners=False)
    return x.permute(0, 2, 3, 1).cpu().numpy()

def normalize01(x: np.ndarray):
    x = x.astype(np.float32)
    x = x - x.min()
    x = x / (x.max() + 1e-6)
    return x

def overlay_rgb_on_frame(frame: np.ndarray, rgb: np.ndarray, alpha: float = 0.45):
    frame = frame.astype(np.float32) / 255.0
    rgb = np.clip(rgb, 0.0, 1.0)
    return np.clip((1 - alpha) * frame + alpha * rgb, 0.0, 1.0)

def colorize_heatmap(heatmap: np.ndarray, cmap_name: str = 'magma'):
    cmap = plt.get_cmap(cmap_name)
    return cmap(normalize01(heatmap))[..., :3]


## Load model and checkpoint

Use the same encoder class and clip geometry as the checkpoint was trained with. For EchoJEPA inference checkpoints, `target_encoder` is the usual state-dict key.

In [ ]:
encoder_ctor = vit.__dict__[MODEL_NAME]
encoder = encoder_ctor(
    img_size=(IMG_SIZE, IMG_SIZE),
    num_frames=NUM_FRAMES,
    tubelet_size=TUBELET_SIZE,
    patch_size=PATCH_SIZE,
    use_rope='rope' in MODEL_NAME,
    uniform_power=True,
).to(device).eval()

ckpt = load_encoder_from_checkpoint(encoder, CHECKPOINT_PATH, CHECKPOINT_KEY)
print('encoder layers:', encoder.get_num_layers())
print('embed dim:', encoder.embed_dim)
print('pos embed exists:', encoder.pos_embed is not None)

## Load one clip

This cell uses `VideoDataset` when available so the sampled `clip_indices` mirror the repository's clip sampling logic. If the source is a single MP4 path, the notebook writes a tiny one-line CSV in the system temp directory and lets `VideoDataset` sample from that.

In [ ]:
raw_clip, clip_indices, label = load_raw_clip_from_source(VIDEO_SOURCE, SAMPLE_INDEX)
print('raw_clip shape:', raw_clip.shape)  # T, H, W, C
print('label:', label)
print('clip_indices:', np.asarray(clip_indices))

clip = prepare_clip_for_model(raw_clip, IMG_SIZE).to(device)
print('model clip shape:', tuple(clip.shape))  # B, C, T, H, W

frame_indices = np.asarray(clip_indices)
tubelet_frame_indices = frame_indices.reshape(-1, TUBELET_SIZE).mean(axis=1) if len(frame_indices) >= TUBELET_SIZE else frame_indices

## Extract patch tokens and reshape into a spatiotemporal grid

The model returns patch tokens in flattened form. We recover `[time, height, width, dim]` using the known patch geometry so that the temporal axis remains explicit.

In [ ]:
with torch.inference_mode():
    tokens = encoder(clip)

print('tokens shape:', tuple(tokens.shape))  # B, N, D
token_grid = reshape_tokens_to_grid(tokens, clip, PATCH_SIZE, TUBELET_SIZE)[0].detach().cpu().numpy()
print('token_grid shape:', token_grid.shape)  # T', H', W', D

T_p, H_p, W_p, D = token_grid.shape
print({'tubelets': T_p, 'patch_h': H_p, 'patch_w': W_p, 'dim': D})

## PCA color map over tokens

This is the DINO-style part: fit PCA to the token cloud, then use the first three principal components as RGB channels. The result is a semantic color map that often highlights anatomical regions and motion boundaries without any labels.

In [ ]:
token_bank = token_grid.reshape(-1, token_grid.shape[-1])
pca = fit_rgb_pca(token_bank)
rgb_grid = pca_to_rgb(token_grid, pca)
rgb_up = upsample_grid(rgb_grid, IMG_SIZE)
frames = raw_clip.astype(np.float32) / 255.0
rgb_frames = np.repeat(rgb_up, TUBELET_SIZE, axis=0)[: len(frames)]
overlay_frames = np.stack([overlay_rgb_on_frame(frames[t], rgb_frames[t], alpha=0.5) for t in range(len(frames))])

n_show = min(len(frames), 8)
show_idx = np.linspace(0, len(frames) - 1, n_show).astype(int)
fig, axes = plt.subplots(2, n_show, figsize=(2.8 * n_show, 6))
if n_show == 1:
    axes = np.array([[axes[0]], [axes[1]]])
for col, t in enumerate(show_idx):
    axes[0, col].imshow(frames[t])
    axes[0, col].set_title(f'raw t={t}')
    axes[0, col].axis('off')
    axes[1, col].imshow(overlay_frames[t])
    axes[1, col].set_title(f'PCA overlay t={t}')
    axes[1, col].axis('off')
plt.tight_layout()
plt.show()

## Temporal analysis: change maps and embedding trajectory

A video model should not only tell us *what* is present, but also *how it changes*. Two simple but effective temporal views are:

- **Change maps**: the L2 norm of the difference between consecutive tubelet grids, upsampled back to image space. This highlights regions where the representation shifts most over time.
- **Trajectory plots**: mean-pool tokens per tubelet, project those pooled vectors to 2D with PCA, and plot them in time order. A loop, drift, or sudden jump can be more informative than the final embedding alone.

In [ ]:
# --- temporal change map ---
delta_grid = np.linalg.norm(np.diff(token_grid, axis=0), axis=-1) if T_p > 1 else np.zeros((1, H_p, W_p))
delta_grid = np.concatenate([delta_grid[:1], delta_grid], axis=0)  # align shape with tubelets
delta_up = upsample_grid(delta_grid[..., None], IMG_SIZE)[..., 0]  # to H x W
delta_frames = np.repeat(delta_up, TUBELET_SIZE, axis=0)[: len(frames)]

fig, axes = plt.subplots(2, n_show, figsize=(2.8 * n_show, 6))
if n_show == 1:
    axes = np.array([[axes[0]], [axes[1]]])
for col, t in enumerate(show_idx):
    axes[0, col].imshow(frames[t])
    axes[0, col].set_title(f'raw t={t}')
    axes[0, col].axis('off')
    axes[1, col].imshow(frames[t])
    axes[1, col].imshow(delta_frames[t], cmap='magma', alpha=0.45)
    axes[1, col].set_title(f'change map t={t}')
    axes[1, col].axis('off')
plt.tight_layout()
plt.show()

# --- temporal trajectory ---
tubelet_repr = token_grid.mean(axis=(1, 2))  # T', D
traj_pca = PCA(n_components=2, random_state=0).fit_transform(tubelet_repr)
time_color = np.linspace(0, 1, len(traj_pca))

fig, ax = plt.subplots(figsize=(6, 6))
sc = ax.scatter(traj_pca[:, 0], traj_pca[:, 1], c=time_color, cmap='viridis', s=70)
ax.plot(traj_pca[:, 0], traj_pca[:, 1], color='gray', linewidth=1, alpha=0.7)
for i in range(len(traj_pca) - 1):
    ax.annotate('', xy=traj_pca[i + 1], xytext=traj_pca[i], arrowprops=dict(arrowstyle='->', color='gray', lw=1))
for i, (x, y) in enumerate(traj_pca):
    ax.text(x, y, str(int(tubelet_frame_indices[i]) if i < len(tubelet_frame_indices) else i), fontsize=8, alpha=0.85)
ax.set_title('Tubelet-level embedding trajectory')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
plt.colorbar(sc, ax=ax, label='normalized time')
plt.tight_layout()
plt.show()

motion_scalar = np.linalg.norm(np.diff(tubelet_repr, axis=0), axis=1) if len(tubelet_repr) > 1 else np.zeros(1)
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(np.arange(len(motion_scalar)), motion_scalar, marker='o')
ax.set_title('Representation change between consecutive tubelets')
ax.set_xlabel('tubelet index')
ax.set_ylabel('L2 distance')
plt.tight_layout()
plt.show()

## Notes on extending this notebook

- Fit PCA on a **shared token bank** from several clips if you want colors that are comparable across videos.
- For fine-grained dynamics, reduce `TUBELET_SIZE` or sample clips with smaller temporal stride.
- If you want to mimic the classic DINO visualization even more closely, you can also colorize **frame-pooling vectors** or attention maps from intermediate blocks using the same PCA basis.
- For ultrasound specifically, the temporal trajectory often separates cycles, valve motion, or contractility changes better than a single pooled embedding.